# Sabak Voice Lab
تجربة أصوات جاهزة للعبة عمر، بدون تسجيل صوتك. شغّل الخلايا بالترتيب.
قبلها: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1) تثبيت الأدوات (3 إلى 5 دقائق)
!pip -q install coqui-tts faster-whisper
import os
os.environ["COQUI_TOS_AGREED"]="1"   # ترخيص XTTS غير تجاري: للتجربة فقط
print("تم")

In [ ]:
# 2) تحميل النموذج وعرض الأصوات الجاهزة
import torch
from TTS.api import TTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda" if torch.cuda.is_available() else "cpu")
SPK = list(tts.speakers)
print(len(SPK), "صوت:")
print(SPK)

In [ ]:
# 3) جرّب كل صوت بنفس الجملة العربية، واستمع (يأخذ دقيقتين تقريباً)
from IPython.display import Audio, display
TEST = "بابا، ليش يقولون مرض؟ مثل الحمى؟ أنا خايف."
os.makedirs("demo", exist_ok=True)
for s in SPK[:20]:
    try:
        tts.tts_to_file(text=TEST, speaker=s, language="ar", file_path=f"demo/{s}.wav")
        print(s); display(Audio(f"demo/{s}.wav"))
    except Exception as e:
        print("فشل", s, e)

In [ ]:
# 4) اختر الأصوات بأسمائها من القائمة فوق، وغيّر الجمل كما تريد
VOICES = {
  "omar":   "Claribel Dervla",   # غيّرها لاسم صوت أعجبك
  "father": "Andrew Chipper",
  "mother": "Daisy Studious",
}
LINES = [
 ("omar",   "omar_01",   "ماما! ماما! أنا لبست وحدي! شوفي، حتى الحذاء!"),
 ("omar",   "omar_03",   "لا! لا يا بابا! أنا ما أبي! أنا بكون مؤدب، والله!"),
 ("omar",   "omar_04",   "سبعة... ثمانية... تسعة... عشرة... بابا؟"),
 ("father", "father_01", "لا تخاف يا عمر، أنا معك. تذكّر المعلم."),
 ("mother", "mother_01", "عمر؟ يا حبيبي، أنت هنا؟"),
]
os.makedirs("out", exist_ok=True)
for who,k,t in LINES:
    tts.tts_to_file(text=t, speaker=VOICES[who], language="ar", file_path=f"out/{k}.wav")
    print("✓", k); display(Audio(f"out/{k}.wav"))

In [ ]:
# 5) فحص الوضوح: Whisper يكتب ما سمعه. قارنه بالأصل
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if torch.cuda.is_available() else -1)
for who,k,t in LINES:
    r = asr(f"out/{k}.wav", generate_kwargs={"language":"arabic","task":"transcribe"})
    print(k, "\n  الأصل:", t, "\n  سمع  :", r["text"])


In [ ]:
# 6) تنزيل كل الملفات
from google.colab import files
!zip -qr voices.zip out
files.download("voices.zip")